This notebook is for generating "silver label" examples using the trained span identification and technique classification models in order to train a lighter weight model. The raw news article data pre-adding silver labels is from the English-only subset of the Common Crawl News dataset. Once run through the existing models to get "silver labels," we use these examples to train a xx model to be used in our Chrome extension.

In [10]:
import pandas as pd
from pathlib import Path
from datasets import load_dataset
import os
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification
import json
from tqdm.auto import tqdm

In [11]:
#Identify base directory to ensure portability
BASE_DIR = Path.cwd().resolve().parent
MODELS_DIR = BASE_DIR / "models"
interim_dir = BASE_DIR / "data" / "interim"
output_file = interim_dir / "news_with_labels.csv"
DATA_PATH = BASE_DIR / "data" / "processed" / "semeval_tc_cleaned.csv"

SI_DIR = MODELS_DIR / "semeval_roberta_scanner"
SI_SPEC_DIR = MODELS_DIR / "semeval_roberta_scanner_specialist"
TC_DIR = MODELS_DIR / "semeval_roberta_classifier"

SI_MODEL_PATH = f"{os.fspath(SI_DIR.absolute())}"
SI_SPEC_PATH = f"{os.fspath(SI_SPEC_DIR.absolute())}"
TC_MODEL_PATH = f"{os.fspath(TC_DIR.absolute())}"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [12]:
#Run training notebooks if models are missing
REQUIRED_FILES = ["config.json", "model.safetensors"]

def model_exists(path):
    path = Path(path)
    has_weights = any(path.glob("*.bin")) or any(path.glob("*.safetensors"))
    return has_weights

if not model_exists(SI_MODEL_PATH):
    print("SI Model missing. Running training notebook...")
    %run 4.1-fp-semeval-si-modeling.ipynb
if not model_exists(TC_MODEL_PATH):
    print("TC Model missing. Running training notebook...")
    %run 4.2-fp-semeval-tc-modeling.ipynb

In [13]:
#Load Base SI Model (RoBERTa token-classifier for span detection)
print(f"Loading Base SI Model from: {SI_MODEL_PATH}...")
si_tokenizer = AutoTokenizer.from_pretrained(SI_MODEL_PATH)
si_model = AutoModelForTokenClassification.from_pretrained(SI_MODEL_PATH, local_files_only=True).to(device)
si_model.eval()

#Load Specialist SI Model
print(f"Loading Specialist SI Model from: {SI_SPEC_PATH}...")
si_spec_model = AutoModelForTokenClassification.from_pretrained(SI_SPEC_PATH, local_files_only=True).to(device)
si_spec_model.eval()

Loading Base SI Model from: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Specialist SI Model from: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner_specialist...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForTokenClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

In [14]:
#Load TC Model (Technique Classification)
tc_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
tc_model = AutoModelForSequenceClassification.from_pretrained(TC_MODEL_PATH).to(device)
tc_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50267, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.2, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [15]:
#Set thresholds for each technique
OPTIMIZED_THRESHOLDS = {
    'Appeal_to_Authority': 0.60,
    'Appeal_to_fear-prejudice': 0.50,
    'Bandwagon_Reductio_ad_hitlerum': 0.10,
    'Black-and-White_Fallacy': 0.20,
    'Causal_Oversimplification': 0.20,
    'Doubt': 0.35,
    'Exaggeration_Minimisation': 0.40,
    'Flag-Waving': 0.45,
    'Loaded_Language': 0.40,
    'Name_Calling_Labeling': 0.55,
    'Repetition': 0.40,
    'Slogans': 0.30,
    'Thought-terminating_Cliches': 0.15,
    'Whataboutism_Straw_Men_Red_Herring': 0.15
}

In [16]:
def run_pipeline_batched(texts):
    """
    Processes a list of texts through the SI -> Cascade -> TC pipeline.
    """
    #1. SI Tokenization (Batch)
    inputs = si_tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        return_offsets_mapping=True
    ).to(device)

    offsets_batch = inputs.pop("offset_mapping")

    with torch.no_grad():
        #2. SI Model Inference (Batch)
        base_outputs = si_model(**inputs)
        base_preds_batch = torch.argmax(base_outputs.logits, dim=-1)

        # 3. Specialist Model Inference (Batch)
        spec_outputs = si_spec_model(**inputs)
        spec_probs_batch = F.softmax(spec_outputs.logits, dim=-1)
        propaganda_prob_batch = spec_probs_batch[:, :, 1]

    #4. Apply Cascade Logic & Extract Spans for each item in batch
    batch_final_results = []

    for i in range(len(texts)):
        text = texts[i]
        base_preds = base_preds_batch[i]
        prop_probs = propaganda_prob_batch[i]
        offsets = offsets_batch[i]

        #Merge predictions
        final_preds = base_preds.clone()
        mask = (base_preds == 0) & (prop_probs > 0.5)
        final_preds[mask] = 1

        #Extract spans
        predicted_spans = []
        current_span = None
        for j, pred in enumerate(final_preds):
            label = pred.item()
            start, end = offsets[j]
            if start == end: continue
            if label in [1, 2]:
                if current_span is None:
                    current_span = [start.item(), end.item()]
                else:
                    current_span[1] = end.item()
            elif current_span:
                predicted_spans.append(tuple(current_span))
                current_span = None
        if current_span: predicted_spans.append(tuple(current_span))

        #5. Technique Classification (TC) for extracted spans
        article_results = []
        for span in predicted_spans:
            span_text = text[span[0]:span[1]].strip()
            if not span_text: continue

            tc_inputs = tc_tokenizer(span_text, return_tensors="pt", truncation=True, padding=True).to(device)
            with torch.no_grad():
                tc_logits = tc_model(**tc_inputs).logits
                probs = torch.sigmoid(tc_logits)[0]

            found_techniques = []
            for class_id, prob in enumerate(probs):
                tech_name = tc_model.config.id2label[class_id]
                if prob.item() >= OPTIMIZED_THRESHOLDS.get(tech_name, 0.5):
                    found_techniques.append(tech_name)

            if not found_techniques:
                found_techniques.append(tc_model.config.id2label[torch.argmax(probs).item()])

            for tech in found_techniques:
                article_results.append({"span": tuple(span), "technique": tech})

        batch_final_results.append(article_results)

    return batch_final_results

In [17]:
#Load `news_with_labels.csv` if it already exists; otherwise, run the labeling pipeline and save the result
if output_file.exists():
    news = pd.read_csv(output_file, names=['text', 'propaganda'], header=0, on_bad_lines='skip')
    news["propaganda"] = news["propaganda"].apply(json.loads)
    display(news)
else:
    print("Cache not found. Downloading the data and running the RoBERTa labeling pipeline (this will take time)...")
    #Load the news article dataset
    news = load_dataset("vblagoje/cc_news", split="train")
    news = news.to_pandas()

    #Constrain to only the text, as that's the only input our extension will be given
    #And take a random sample of articles because the dataset is unreasonably large
    news = news[['text']].sample(n=4000, random_state=42).reset_index(drop=True)
    display(news)

    #Use run pipeline function to get predicted propaganda spans and labels from all the text
    print("Running pipeline...")
    BATCH_SIZE = 32
    all_predictions = []
    texts_to_process = news['text'].tolist()

    for i in tqdm(range(0, len(texts_to_process), BATCH_SIZE)):
        batch = texts_to_process[i : i + BATCH_SIZE]
        batch_results = run_pipeline_batched(batch)
        all_predictions.extend(batch_results)

    news['propaganda'] = all_predictions
    display(news)

    #Save the dataframe so it can be reused later
    print("Saving DataFrame...")
    news_to_save = news.copy()
    news_to_save["propaganda"] = news_to_save["propaganda"].apply(json.dumps)

    news_to_save.to_csv(output_file, index=False)
    print(f"Silver labels saved to {output_file}")

Cache not found. Downloading the data and running the RoBERTa labeling pipeline (this will take time)...


,text
0,Nashik : Indore Infoline Pvt. Ltd has organise...
1,South-East Governors on Monday re-assured Ndig...
2,The two teenagers that were arrested in connec...
3,"CHARLOTTE, North Carolina (Reuters) - Jason Da..."
4,Donald Trump’s baser instincts served him well...
...,...
3995,April 6 Global demand for air travel rose 4.8 ...
3996,Oklahoma health officials say a wild bat found...
3997,"PRATT, Kan. (AP) — Funeral services have been ..."
3998,"LINCOLN, Neb. (AP) _ Nebraska’s longest-servin..."


Running pipeline...


  0%|          | 0/125 [00:00<?, ?it/s]

,text,propaganda
0,Nashik : Indore Infoline Pvt. Ltd has organise...,[]
1,South-East Governors on Monday re-assured Ndig...,"[{'span': (700, 707), 'technique': 'Bandwagon_..."
2,The two teenagers that were arrested in connec...,[]
3,"CHARLOTTE, North Carolina (Reuters) - Jason Da...","[{'span': (203, 211), 'technique': 'Bandwagon_..."
4,Donald Trump’s baser instincts served him well...,"[{'span': (15, 30), 'technique': 'Causal_Overs..."
...,...,...
3995,April 6 Global demand for air travel rose 4.8 ...,[]
3996,Oklahoma health officials say a wild bat found...,[]
3997,"PRATT, Kan. (AP) — Funeral services have been ...",[]
3998,"LINCOLN, Neb. (AP) _ Nebraska’s longest-servin...","[{'span': (627, 632), 'technique': 'Appeal_to_..."


Saving DataFrame...
Silver labels saved to /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/data/interim/news_with_labels.csv


In [18]:
#Add in gold standard labels (official labeled spans from SemEval, the original dataset)
se = pd.read_csv(Path("..") / "data" / "processed" / "semeval_tc_cleaned.csv")
se.head(2)

,article_id,text_content,span_text,start_char,end_char,sentiment,punct_count,lexical_diversity,Appeal_to_Authority,Appeal_to_fear-prejudice,...,Causal_Oversimplification,Doubt,Exaggeration_Minimisation,Flag-Waving,Loaded_Language,Name_Calling_Labeling,Repetition,Slogans,Thought-terminating_Cliches,Whataboutism_Straw_Men_Red_Herring
0,111111111,Next plague outbreak in Madagascar could be 's...,appeared,149,157,0.00,0,1.0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,111111111,Next plague outbreak in Madagascar could be 's...,The next transmission could be more pronounced...,265,323,0.25,0,1.0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [19]:
def convert_to_news_format(span_df):
    #1. Define the technique columns
    tech_cols = [
        'Appeal_to_Authority', 'Appeal_to_fear-prejudice', 'Bandwagon_Reductio_ad_hitlerum',
        'Black-and-White_Fallacy', 'Causal_Oversimplification', 'Doubt',
        'Exaggeration_Minimisation', 'Flag-Waving', 'Loaded_Language',
        'Name_Calling_Labeling', 'Repetition', 'Slogans',
        'Thought-terminating_Cliches', 'Whataboutism_Straw_Men_Red_Herring'
    ]

    #2. Helper function to create the 'propaganda' list for a single span row
    def row_to_dict(row):
        found_techs = []
        for col in tech_cols:
            if row[col] == 1:
                found_techs.append({
                    'span': [int(row['start_char']), int(row['end_char'])],
                    'technique': col
                })
        return found_techs

    #Apply helper to create a temporary list column
    span_df['temp_list'] = span_df.apply(row_to_dict, axis=1)

    #3. Group by Article and aggregate
    #'text' remains the same for every row in a group, so we take 'first'
    #'propaganda' becomes a flat list of all spans found across all rows for that article
    new_df = span_df.groupby('article_id').agg({
        'text_content': 'first',
        'temp_list': 'sum'  # This flattens the lists of dictionaries
    }).reset_index()

    #4. Final Rename and formatting
    new_df = new_df.rename(columns={'text_content': 'text', 'temp_list': 'propaganda'})

    return new_df[['text', 'propaganda']]

news_se = convert_to_news_format(se)
news_se.head()

,text,propaganda
0,Next plague outbreak in Madagascar could be 's...,"[{'span': [149, 157], 'technique': 'Doubt'}, {..."
1,US bloggers banned from entering UK\n\nTwo pro...,"[{'span': [191, 219], 'technique': 'Slogans'},..."
2,Kate Steinle's death at the hands of a Mexican...,"[{'span': [259, 279], 'technique': 'Loaded_Lan..."
3,U.S. judge frees Indonesian immigrant held by ...,"[{'span': [1705, 1824], 'technique': 'Appeal_t..."
4,Here are all the sexual misconduct accusations...,"[{'span': [658, 700], 'technique': 'Loaded_Lan..."


In [20]:
#Merge gold spans with silver ones
news = pd.concat([news, news_se], axis=0).reset_index(drop=True)
display(news)

,text,propaganda
0,Nashik : Indore Infoline Pvt. Ltd has organise...,[]
1,South-East Governors on Monday re-assured Ndig...,"[{'span': (700, 707), 'technique': 'Bandwagon_..."
2,The two teenagers that were arrested in connec...,[]
3,"CHARLOTTE, North Carolina (Reuters) - Jason Da...","[{'span': (203, 211), 'technique': 'Bandwagon_..."
4,Donald Trump’s baser instincts served him well...,"[{'span': (15, 30), 'technique': 'Causal_Overs..."
...,...,...
4352,Altered Election Documents Tied To Florida Dem...,"[{'span': [86, 101], 'technique': 'Loaded_Lang..."
4353,Migrant Caravan Reach Border & Climb Atop Fenc...,"[{'span': [31, 62], 'technique': 'Loaded_Langu..."
4354,Guardian ups its vilification of Julian Assang...,"[{'span': [17, 29], 'technique': 'Loaded_Langu..."
4355,This Guardian Fake News Story Proves That The ...,"[{'span': [0, 68], 'technique': 'Doubt'}, {'sp..."


In [21]:
#Add in Gemini-generated examples based on the SemEval provided examples of each technique
gg = pd.read_csv(Path("..") / "data" / "interim" / "gemini_generated_examples.csv")
gg.head(2)

,text,propaganda
0,Outrage as Donald Trump suggests injecting dis...,"[{""span"":[0,6],""technique"":""Loaded_Language""}]"
1,The senator's vile betrayal of working familie...,"[{""span"":[14,18],""technique"":""Loaded_Language""..."


In [22]:
#Merge with other spans
news = pd.concat([gg, news], axis=0).reset_index(drop=True)
display(news)

,text,propaganda
0,Outrage as Donald Trump suggests injecting dis...,"[{""span"":[0,6],""technique"":""Loaded_Language""}]"
1,The senator's vile betrayal of working familie...,"[{""span"":[14,18],""technique"":""Loaded_Language""..."
2,Brave freedom fighters resist the tyrannical o...,"[{""span"":[0,5],""technique"":""Loaded_Language""},..."
3,The corrupt elites are bleeding this country d...,"[{""span"":[4,10],""technique"":""Loaded_Language""}..."
4,A catastrophic failure of leadership has plung...,"[{""span"":[2,13],""technique"":""Loaded_Language""}..."
...,...,...
5852,Altered Election Documents Tied To Florida Dem...,"[{'span': [86, 101], 'technique': 'Loaded_Lang..."
5853,Migrant Caravan Reach Border & Climb Atop Fenc...,"[{'span': [31, 62], 'technique': 'Loaded_Langu..."
5854,Guardian ups its vilification of Julian Assang...,"[{'span': [17, 29], 'technique': 'Loaded_Langu..."
5855,This Guardian Fake News Story Proves That The ...,"[{'span': [0, 68], 'technique': 'Doubt'}, {'sp..."


In [24]:
#Save dataframe for use later
news.to_csv(Path("..") / "data" / "processed" / "distilled.csv")